# Opis

NOtebook służący przygotowaniu testów do wykoania w slurmie

# Importy

In [3]:
IS_NEW_APPROACH = True
IS_SLURM = False

In [4]:
%load_ext autoreload
%autoreload 2
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import GradientAccumulationScheduler
import yaml
import sys
import tqdm
import wandb
import json
from pyprojroot import here
import argparse
sys.path.append('../') # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.KlejdaGraphAutoencoder import KlejdaGraphAutoencoder
from src.models.KlejdaGAE.KlejdaVariationalGraphAutoencoder import KlejdaVariationalGraphAutoencoder
from src.models.NewGAE.GraphAutoencoder import GraphAutoencoder
from src.models.NewGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from src.other.QuadraticGradientAccumulationScheduler import QuadraticGradientAccumulationScheduler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import os
import sys
current_dir = os.getcwd()
framspy_path = os.path.abspath(os.path.join(current_dir, '..', 'external', 'framspy'))
if framspy_path not in sys.path:
	sys.path.insert(0, framspy_path)
with open("../configs/final_evolution_config.yaml", 'r') as f:
	evolution_config = yaml.safe_load(f)
import frams

frams.init(
    evolution_config['frams_path']
)

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data



In [ ]:
parser = argparse.ArgumentParser(description="Trening modelu na różnych konfiguracjach")

parser.add_argument(
    "--model_type",
    type=str,
    choices=["gae", "vgae"],
    default="gae",
    help="Zwykły 'gae', lub wariacyjny 'vgae'"
)

parser.add_argument(
    "--locality_loss_type",
    type=str,
    choices=["parts_num", "fitness"], #TODO: Dodać jeszcze silimality, jak już będzie szybciej działać
    default="fitness",
    help="Typ locality loss"
)

parser.add_argument(
    "--latent_dim",
    type=int,
    choices=[3, 10, 15, 100],
    default=3,
    help="Rozmiar przestrzeni ukrytej"
)

parser.add_argument(
    "--layers_config",
    type=str,
    choices=["small", "large"],
    default="small",
    help="Wariant głębokości sieci: 'small' lub 'large'"
)
parser.add_argument(
	"--exp_id",
	type=int,
	default=0,
	help="ID eksperymentu z siatki"
)
parser.add_argument("--run_name", type=str, default="slurm", help="Własna nazwa eksperymentu w WandB")
parser.add_argument("--group", type=str, default="slurm", help="Grupa eksperymentów w WandB")

args = parser.parse_args()

# Przetwarzanie

## Załadowanie danych i przygotowanie do przetwarzania

In [7]:
# dataset = FramsticksDummyDataset(num_samples=1000)
torch.set_float32_matmul_precision('medium')
genotypes = []
with open("../results/sampled_best_individuals_new_mini_merged.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line.strip())
        genotypes.append(obj)

if IS_SLURM:
    project_dir = Path("/home/inf151848/MasterThesisPP")
else:
    project_dir = here()

configs_dir = project_dir / 'configs'
if IS_NEW_APPROACH:
	config_gae_path = configs_dir / 'gae_config_final.yaml'
else:
	config_gae_path = configs_dir / 'klejda_gae_config.yaml'

with open(config_gae_path) as f:
    config = yaml.safe_load(f)

config["model_type"] = args.model_type
config["latent_dim"] = args.latent_dim
config["layers_config"] = args.layers_config
config["locality_loss_type"] = args.locality_loss_type
config["exp_id"] = args.exp_id
IS_VGAE = config["model_type"] == "vgae"

# Wczytanie dodatkowego configu od warstw modelu
config_layers = configs_dir
if config['layers_config'] == "small":
	config_layers = config_layers / "layers_small.yaml"
else:
	config_layers = config_layers / "layers_large.yaml"
with open(config_layers, "r") as f:
	config_layers_dict = yaml.safe_load(f)

for k in config_layers_dict.keys():
	config[k] = config_layers_dict[k]

dataset = FramsticksGraphDataset(genotypes,config["max_nodes"])

dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size
print("train_size:", train_size)
print("val_size:", val_size)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Akumulator, mający za zadanie zmieniać rozmiar batcha wraz z postępującym uczeniem modelu
# W początkowej fazie rozmiar jest mniejszy, wtedy bowiem model lepiej naucza się jak torzyć macierze X i A
# w późniejszym etapie rozmiar batcha wzrasta, aby skupić się poprawnym zmniejszeniu locality loss

if IS_NEW_APPROACH and config['use_accumulator']:
	accumulator = QuadraticGradientAccumulationScheduler(1,16,5,config['max_epochs'])
else:
	accumulator = GradientAccumulationScheduler(scheduling={
	    0:1,
	})

train_dataloader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=0,
    persistent_workers=False,
	drop_last=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=0,
    persistent_workers=False,
	drop_last=True
)

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\witek\_netrc.


train_size: 47809
val_size: 11953


wandb: Currently logged in as: witekadrian7 (witekadrian7-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## GAE

In [44]:
if not IS_VGAE:
	if IS_NEW_APPROACH:
		modelGAE = GraphAutoencoder(config, frams_module=frams)
	else:
		modelGAE = KlejdaGraphAutoencoder(config, frams_module=frams)
	wandb.finish()

wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading summary
wandb:  View run GAE_klejda_test at: https://wandb.ai/witekadrian7-none/Framsticks-GAE/runs/i00bqyrc
wandb:  View project at: https://wandb.ai/witekadrian7-none/Framsticks-GAE
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: checkpoints\wandb\run-20260903_202357-i00bqyrc\logs


In [45]:
if not IS_VGAE:
	wandb_logger = WandbLogger(project="Framsticks-MasterThesis", name=args.run_name, save_dir = config["save_dir"],tags=[args.model_type, args.layers_config, f"lat_{args.latent_dim}", f"loc_{args.locality_loss_type}"])
	wandb_logger.log_hyperparams(config)
	trainer = pl.Trainer(
	    max_epochs=config['max_epochs'],
	    logger=wandb_logger,
		callbacks=[accumulator],
	    enable_progress_bar=False,
		log_every_n_steps=10,
	    accelerator="auto",
	    devices=1
	)
	trainer.fit(modelGAE, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
	wandb.finish()

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
wandb: setting up run 8g138svs
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260903_202546-8g138svs
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run GAE_klejda_test
wandb:  View project at https://wandb.ai/witekadrian7-none/Framsticks-GAE
wandb:  View run at https://wandb.ai/witekadrian7-none/Framsticks-GAE/runs/8g138svs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES:

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ KlejdaEncoder  │ 70.9 K │ train │     0 │
│ 1 │ fc_z             │ Linear         │    195 │ train │     0 │
│ 2 │ decoder_a        │ KlejdaDecoderA │ 38.0 K │ train │     0 │
│ 3 │ decoder_x        │ KlejdaDecoderX │ 30.9 K │ train │     0 │
│ 4 │ criterion        │ MSELoss        │      0 │ train │     0 │
└───┴──────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 139 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 139 K                                                                                                
Total estimated model params size (MB): 0.560                                                                      
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## VGAE

In [8]:
if IS_VGAE:
	if IS_NEW_APPROACH:
		modelVGAE = VariationalGraphAutoencoder(config, frams_module=frams)
	else:
		modelVGAE = KlejdaVariationalGraphAutoencoder(config, frams_module=frams)
	wandb.finish()

In [9]:
if IS_VGAE:
	wandb_logger = WandbLogger(project="Framsticks-MasterThesis", name=args.run_name, save_dir = config["save_dir"], tags=[args.model_type, args.layers_config, f"lat_{args.latent_dim}", f"loc_{args.locality_loss_type}"])
	wandb_logger.log_hyperparams(config)
	trainer = pl.Trainer(
	    max_epochs=config['max_epochs'],
	    logger=wandb_logger,
		callbacks=[accumulator],
	    enable_progress_bar=False,
		log_every_n_steps=10,
	    accelerator="auto",
	    devices=1
	)

	trainer.fit(modelVGAE, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
	wandb.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: setting up run pp7o4iwh
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260906_192026-pp7o4iwh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run VGAE_new_test
wandb:  View project at https://wandb.ai/witekadrian7-none/Framsticks-VGAE
wandb:  View run at https://wandb.ai/witekadrian7-none/Framsticks-VGAE/runs/p

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │  2.1 M │ train │     0 │
│ 1 │ fc_mu            │ Linear   │  1.9 K │ train │     0 │
│ 2 │ fc_logvar        │ Linear   │  1.9 K │ train │     0 │
│ 3 │ decoder_a        │ DecoderA │ 93.7 K │ train │     0 │
│ 4 │ decoder_x        │ DecoderX │  562 K │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11.076                                                                     
Modules in train mode: 65                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\pytorch_lightning\utilities\data.py:79: Trying to
infer the `batch_size` from an ambiguous collection. The batch size we found is 256. To avoid any miscalculations, 
use `self.log(..., batch_size=batch_size)`.

`Trainer.fit` stopped: `max_epochs=100` reached.


wandb: updating run metadata
wandb: uploading config.yaml; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:                      epoch ▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇████
wandb: train/locality_correlation ▁▁▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇██████████
wandb:               train/loss_A █▆▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:              train/loss_KL ▁▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇████████████████
wandb:               train/loss_X █▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        train/loss_locality █▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:           train/loss_total █▅▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:          train/metric_A_F1 ▁▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇███████████
wandb:    train/metric_A_fp_ratio ██▇▇▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
wandb:      train/metric_A_g_mean ▁▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████████████
wandb:                        +16 ...
wandb: 
wandb: Run summary:
wandb:                      epoch 99
wandb: train/locality_correlati